# Cat & Dog Image Classifier 🐱🐶

A reproducible TensorFlow/Keras CNN for the freeCodeCamp Machine Learning with Python challenge. Train in Google Colab; benchmark results are not precomputed.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile, os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

tf.keras.utils.set_random_seed(42)
IMG_HEIGHT = IMG_WIDTH = 150
batch_size = 32
epochs = 25
archive = Path("cats_and_dogs.zip")
if not archive.exists():
    urlretrieve("https://cdn.freecodecamp.org/project-data/cats-and-dogs/cats_and_dogs.zip", archive)
if not Path("cats_and_dogs/train").exists():
    with zipfile.ZipFile(archive) as z:
        z.extractall(".")
PATH = "cats_and_dogs"
train_dir = os.path.join(PATH, "train")
validation_dir = os.path.join(PATH, "validation")
test_dir = os.path.join(PATH, "test")


## Cell 3 — Normalize and load images
The test directory has 50 unlabeled images. Selecting `classes=['test']` prevents mistakenly finding zero samples.

In [ ]:
train_image_generator = ImageDataGenerator(rescale=1./255)
validation_image_generator = ImageDataGenerator(rescale=1./255)
test_image_generator = ImageDataGenerator(rescale=1./255)
train_data_gen = train_image_generator.flow_from_directory(
    train_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=batch_size, class_mode="binary")
val_data_gen = validation_image_generator.flow_from_directory(
    validation_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=batch_size,
    class_mode="binary", shuffle=False)
test_data_gen = test_image_generator.flow_from_directory(
    PATH, classes=["test"], target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=batch_size, class_mode=None, shuffle=False)
assert test_data_gen.samples == 50
print(train_data_gen.class_indices)


## Cells 4–6 — Show training samples and apply six augmentations

In [ ]:
def plotImages(images_arr, probabilities=None):
    images_arr = list(images_arr)
    fig, axes = plt.subplots(len(images_arr), 1, figsize=(5, len(images_arr)*2.5))
    if len(images_arr) == 1:
        axes = [axes]
    for i, (image, ax) in enumerate(zip(images_arr, axes)):
        ax.imshow(image)
        ax.axis("off")
        if probabilities is not None:
            p = float(probabilities[i])
            ax.set_title(f"{p*100:.1f}% dog" if p > .5 else f"{(1-p)*100:.1f}% cat")
    plt.tight_layout()
    plt.show()

sample_images, _ = next(train_data_gen)
plotImages(sample_images[:5])
train_image_generator = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=.12,
    height_shift_range=.12, shear_range=.12,
    zoom_range=.15, horizontal_flip=True)
train_data_gen = train_image_generator.flow_from_directory(
    train_dir, batch_size=batch_size, target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary")
plotImages([train_data_gen[0][0][0] for _ in range(5)])


## Cell 7 — Build the convolutional network

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input((IMG_HEIGHT, IMG_WIDTH, 3)),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


## Cell 8 — Train and validate
A learning-rate scheduler and early stopping limit overfitting.

In [ ]:
history = model.fit(
    train_data_gen, steps_per_epoch=len(train_data_gen), epochs=epochs,
    validation_data=val_data_gen, validation_steps=len(val_data_gen),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3,
                                              factor=.5, min_lr=1e-5)
    ])


## Cell 9 — Plot learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for metric, ax in zip(("accuracy", "loss"), axes):
    ax.plot(history.history[metric], label="train")
    ax.plot(history.history["val_"+metric], label="validation")
    ax.set_title(metric.capitalize())
    ax.set_xlabel("Epoch")
    ax.legend()
plt.show()


## Cell 10 — Predict all 50 images in deterministic order

In [ ]:
test_data_gen.reset()
probabilities = model.predict(test_data_gen, steps=len(test_data_gen), verbose=0).ravel().tolist()
test_images = np.concatenate([test_data_gen[i] for i in range(len(test_data_gen))], axis=0)
assert len(probabilities) == 50
plotImages(test_images, probabilities)


## Cell 11 — Evaluate against labels published in the official starter notebook
The target is **at least 63%**. These labels are used for evaluation only.

In [ ]:
answers = [1,0,0,1,0,0,0,0,1,1,0,1,0,1,0,1,1,0,1,1,0,0,
           1,1,1,1,1,0,0,0,0,0,1,1,0,1,1,1,1,0,1,0,1,1,0,0,0,0,0,0]
accuracy = float(np.mean(np.rint(probabilities).astype(int) == np.asarray(answers))*100)
print(f"Correctly classified {accuracy:.2f}% of the 50 test images")
print("PASS" if accuracy >= 63 else "NOT YET PASSING — tune and train again")
